# L3 on the PC · Calibrating the 4-bit 3B's confidence and setting its hand-off to the 7B

Task **L3** (roadmap §7), objective **O4**. Reads the Kaggle run `l3-4bit-3b-handoff` (saved in `L3-results/full/L3/`) and the new 7B's saved answers to the same 10,675 questions (L0 step 2, `L0-step2-results/new-partA` and `new-partB`). No model runs here: the hand-off is worked out from answers both models already gave. Writes `L3-results/L3/handoff/`.

**The idea:** the 3B answers every question first. When it is sure, its answer stands; when it isn't, the question goes to the 7B. That saves time only if the 3B's confidence says when it is right, so the confidence is checked before a threshold is set.

**Rule, fixed before the run (23 Nov), the same text as in the Kaggle notebook**
- *Confidence of an answer:* the geometric mean of the probabilities of its tokens before the first end token (0 for an empty answer).
- *Halves:* a question goes to the choosing half if the md5 of its ID is even, otherwise to the reporting half. Everything is chosen on the first and reported on the second.
- *Temperature:* the one of 10 (0.5 to 8) with the lowest log loss of confidence against correctness on the choosing half. A choice at either end of the list is flagged: the best temperature may lie beyond it.
- *Checks on the reporting half:* the expected calibration error (ECE, 15 equal-width bins) at temperature 1 and at the chosen one, and the risk–coverage curve with its area (RC-AUC).
- *Threshold:* on the choosing half, the threshold that hands the fewest questions to the 7B while the combined accuracy, as the mean of the six tasks' accuracies, stays at least the 7B alone's mean minus 1.0 percentage point.
- *Reported on the reporting half:* the share the 3B answers, the combined accuracy, the 7B alone, the 3B alone, the expected seconds per question, all also on E5's clean set (its 186 copied questions dropped).

**Why this rule, and what else could be used**
- *Geometric mean of the token probabilities* (the roadmap's choice, from Grace). It is the model's own probability for its answer, scaled for length, so an answer naming three options isn't marked unsure just for being longer. Other options:
  - *The product of the probabilities*: it does mark longer answers unsure.
  - *The entropy of the first token*: it ignores the rest of the answer.
  - *A confidence the model states in words*: TimeSpot found these poorly calibrated in open VLMs.
  - *Agreement among several sampled answers* (self-consistency): it costs several runs per question, which defeats a hand-off meant to save time.
- *Temperature scaling* (Guo et al., 2017) is the usual first fix for overconfidence. It has one number to fit, so half the benchmark is plenty. Histogram binning or isotonic regression fit more freely and need more data. They stay options if one temperature leaves the ECE high.
- *A 1-point margin:* the hand-off should cost almost no accuracy against the 7B. One point is about one standard error of the six-task mean on half the benchmark; this notebook prints that error. A single threshold for all tasks keeps the rule simple. Per-task thresholds stay an option if one task falls far below the 7B.
- *The mean of the six tasks' accuracies*, not the share of all questions answered right. Counting (BDC) holds a third of the questions, and the paper reports the tasks separately.

**How to run:** on the PC, in Jupyter (cse499 environment), from `Champion_Task/`, after the Kaggle run's outputs are saved in `L3-results/full/L3/`. Run all; it takes about a minute.

In [1]:
# ── Cell 1: The 3B's answers and token scores, the 7B's saved answers, the halves and E5's clean set ──
import hashlib
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

CT = Path.cwd()                                          # Champion_Task
L3_DIR = CT / "L3-results" / "full" / "L3"
OUT = L3_DIR / "handoff"
SEVEN_B_ANSWERS = [CT / "L0-step2-results" / p / "L0S2" / "L0_step2_new_per_question.csv" for p in ("new-partA", "new-partB")]
FP16_DIR = CT / "L2-step4-results" / "full" / "L2S4"                   # L2 run 5's full run (the 3B in fp16), once saved
FLAGGED = CT / "E5-results" / "step1" / "bench_questions_flagged.csv"
TEMPS = (0.5, 0.75, 1.0, 1.25, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0)
MARGIN, BINS = 1.0, 15                                   # fixed 23 Nov
TASK_ORDER = ["DTR", "BBR", "BDC", "DRE", "DSR", "ORR"]
OUT.mkdir(parents=True, exist_ok=True)

config = json.loads((L3_DIR / "config.json").read_text(encoding="utf-8"))
assert not config["smoke_test"] and config["complete"], "L3's Kaggle run is a smoke test or isn't complete"
speed = json.loads((L3_DIR / "speed.json").read_text(encoding="utf-8"))
with open(L3_DIR / "answers.jsonl", encoding="utf-8") as f:
    stats = {r["id"]: r for r in map(json.loads, f)}

def confidence(stat, t=1.0):
    """The geometric mean of the answer's token probabilities at temperature t (0 for an empty answer): L3's cell 2."""
    if not stat["z"]:
        return 0.0
    k = TEMPS.index(t)
    return math.exp(sum(z / t - l[k] for z, l in zip(stat["z"], stat["lse"])) / len(stat["z"]))

pq = pd.read_csv(L3_DIR / "per_question.csv", usecols=["id", "task", "truth", "answer", "correct"])
seven = pd.concat([pd.read_csv(p) for p in SEVEN_B_ANSWERS]).dropna(subset=["answer"]).drop_duplicates("id").set_index("id")
pq["correct_3b"] = pq["correct"].astype(bool)
pq["correct_7b"] = pq["id"].map(seven["correct"]).fillna(False).astype(bool)
for t in TEMPS:
    pq[f"conf_{t}"] = [confidence(stats[i], t) if i in stats else 0.0 for i in pq["id"]]
assert np.isfinite(pq[[f"conf_{t}" for t in TEMPS]].to_numpy()).all(), \
    "a confidence isn't a number: the Kaggle run overflowed (23 Nov; see its README)"
pq["half"] = ["choose" if int(hashlib.md5(i.encode()).hexdigest(), 16) % 2 == 0 else "report" for i in pq["id"]]
flag = pd.read_csv(FLAGGED)
copied = set(flag.loc[(flag["flag"] == "after-image copied") & (flag["sensor"] == "Optical"), "id"])
pq["clean"] = ~pq["id"].isin(copied)
assert len(pq) == 10675 and len(copied) == 186 and pq["id"].isin(copied).sum() == 186
print(f"✓ {len(pq):,} questions; the 7B answered {pq['id'].isin(seven.index).sum():,}; "
      f"halves {pq['half'].value_counts().to_dict()}; clean {int(pq['clean'].sum()):,}")

✓ 10,675 questions; the 7B answered 10,667; halves {'report': 5362, 'choose': 5313}; clean 10,489


In [2]:
# ── Cell 2: Calibration: the temperature from the choosing half, ECE and risk-coverage on the reporting half ──
def log_loss(c, y):
    c = np.clip(c, 1e-6, 1 - 1e-6)
    return float(-np.mean(y * np.log(c) + (1 - y) * np.log(1 - c)))

def ece(c, y, bins=BINS):
    """Expected calibration error: in each of the equal-width confidence bins, |mean confidence - accuracy|, weighted by
    the bin's share of the answers."""
    b = np.minimum((c * bins).astype(int), bins - 1)
    return float(sum(abs(c[b == k].mean() - y[b == k].mean()) * (b == k).mean() for k in range(bins) if (b == k).any()))

def risk_coverage(c, y):
    """Answers from the most confident down: the error rate among the first k, for every k; the area is its mean."""
    risk = np.cumsum(1 - y[np.argsort(-c, kind="stable")]) / np.arange(1, len(y) + 1)
    return risk, float(risk.mean())

assert abs(ece(np.array([0.9, 0.9, 0.1, 0.1]), np.array([1, 1, 0, 0])) - 0.1) < 1e-9
assert risk_coverage(np.array([0.9, 0.1]), np.array([1, 0]))[1] == 0.25

ch, rp = pq[pq["half"] == "choose"], pq[pq["half"] == "report"]
y_ch, y_rp = ch["correct_3b"].to_numpy(float), rp["correct_3b"].to_numpy(float)
cal = pd.DataFrame([{"temperature": t, "log_loss_choosing": round(log_loss(ch[f"conf_{t}"].to_numpy(), y_ch), 4),
                     "ece_reporting": round(ece(rp[f"conf_{t}"].to_numpy(), y_rp), 4),
                     "rc_auc_reporting": round(risk_coverage(rp[f"conf_{t}"].to_numpy(), y_rp)[1], 4)} for t in TEMPS])
T = float(cal.loc[cal["log_loss_choosing"].idxmin(), "temperature"])
pq["conf"] = pq[f"conf_{T}"]
ch, rp = pq[pq["half"] == "choose"], pq[pq["half"] == "report"]
c1, cT = rp["conf_1.0"].to_numpy(), rp["conf"].to_numpy()
oracle = risk_coverage(y_rp, y_rp)[1]                    # right answers first: the best any confidence could do
calib = {"temperature": T, "at_list_end": T in (TEMPS[0], TEMPS[-1]), "ece_T1": round(ece(c1, y_rp), 4), "ece_chosen_T": round(ece(cT, y_rp), 4),
         "rc_auc_T1": round(risk_coverage(c1, y_rp)[1], 4), "rc_auc_chosen_T": round(risk_coverage(cT, y_rp)[1], 4),
         "rc_auc_random": round(float(1 - y_rp.mean()), 4), "rc_auc_best": round(oracle, 4),
         "mean_confidence_T1": round(float(c1.mean()), 4), "accuracy_3b": round(float(y_rp.mean()), 4)}

def reliability(c, y):
    b = np.minimum((c * BINS).astype(int), BINS - 1)
    return pd.DataFrame([{"bin": f"{k / BINS:.2f}-{(k + 1) / BINS:.2f}", "answers": int((b == k).sum()),
                          "mean_confidence": round(float(c[b == k].mean()), 3), "accuracy": round(float(y[b == k].mean()), 3)}
                         for k in range(BINS) if (b == k).any()])

rel = pd.concat([reliability(c1, y_rp).assign(temperature=1.0), reliability(cT, y_rp).assign(temperature=T)])
risk_T, _ = risk_coverage(cT, y_rp)
rc = pd.DataFrame([{"coverage_%": p, "error_rate_%": round(100 * float(risk_T[max(1, round(p / 100 * len(risk_T))) - 1]), 2)}
                   for p in (10, 20, 30, 40, 50, 60, 70, 80, 90, 100)])
print(cal.to_string(index=False))
print(f"\nChosen temperature {T}: {calib}\n")
print(rc.to_string(index=False))

 temperature  log_loss_choosing  ece_reporting  rc_auc_reporting
        0.50             2.1066         0.5008            0.4092
        0.75             1.4755         0.4528            0.4056
        1.00             1.1569         0.4034            0.4035
        1.25             0.9468         0.3479            0.4004
        1.50             0.7591         0.2643            0.3944
        2.00             0.6299         0.1229            0.3906
        3.00             1.7036         0.3943            0.3961
        4.00             2.4683         0.4063            0.3970
        6.00             3.2599         0.4081            0.3977
        8.00             3.6612         0.4083            0.3980

Chosen temperature 2.0: {'temperature': 2.0, 'at_list_end': False, 'ece_T1': 0.4034, 'ece_chosen_T': 0.1229, 'rc_auc_T1': 0.4035, 'rc_auc_chosen_T': 0.3906, 'rc_auc_random': 0.5916, 'rc_auc_best': 0.2259, 'mean_confidence_T1': 0.8112, 'accuracy_3b': 0.4084}

 coverage_%  error_rate_%

In [3]:
# ── Cell 3: The threshold from the choosing half; the hand-off on the reporting half and its clean part ──
def task_mean(ok, tasks):
    """The mean of the six tasks' accuracies (%)."""
    return float(pd.Series(np.asarray(ok, float)).groupby(np.asarray(tasks)).mean().reindex(TASK_ORDER).mean() * 100)

def combined(df, tau):
    use = df["conf"].to_numpy() >= tau
    return np.where(use, df["correct_3b"].to_numpy(), df["correct_7b"].to_numpy()), float(use.mean())

target = task_mean(ch["correct_7b"], ch["task"]) - MARGIN
candidates = np.unique(np.r_[ch["conf"].to_numpy(), np.inf])     # lowest first: the first that holds hands on the fewest
tau = next(t for t in candidates if task_mean(combined(ch, t)[0], ch["task"]) >= target)

def standard_error(df, col):
    """The standard error of the six-task mean of accuracy (percentage points)."""
    g = df.groupby("task")[col].agg(["mean", "count"]).reindex(TASK_ORDER)
    return float(np.sqrt((g["mean"] * (1 - g["mean"]) / g["count"]).sum()) / len(g) * 100)

t3, t7 = speed["3b_nf4"]["s_per_question_mean"], speed["7b_nf4"]["s_per_question_mean"]

def rows_for(df, label):
    ok, share = combined(df, tau)
    either = df["correct_3b"].to_numpy() | df["correct_7b"].to_numpy()
    return [{"set": label, "method": "7B alone", "share_by_3b_%": 0.0, "accuracy_%": task_mean(df["correct_7b"], df["task"]),
             "s_per_question": t7},
            {"set": label, "method": "3B alone (4 bits)", "share_by_3b_%": 100.0,
             "accuracy_%": task_mean(df["correct_3b"], df["task"]), "s_per_question": t3},
            {"set": label, "method": f"hand-off at {tau:.4f}", "share_by_3b_%": 100 * share,
             "accuracy_%": task_mean(ok, df["task"]), "s_per_question": t3 + (1 - share) * t7},
            {"set": label, "method": "perfect hand-off (either right)", "share_by_3b_%": None,
             "accuracy_%": task_mean(either, df["task"]), "s_per_question": None}]

handoff = pd.DataFrame(rows_for(rp, "reporting half") + rows_for(rp[rp["clean"]], "reporting half, clean")
                       + rows_for(ch, "choosing half")).round(3)
ok_rp, _ = combined(rp, tau)
by_task = pd.DataFrame([{"task": k, "questions": int((rp["task"] == k).sum()),
                         "share_by_3b_%": round(100 * float((rp.loc[rp["task"] == k, "conf"] >= tau).mean()), 1),
                         "7B_%": round(100 * float(rp.loc[rp["task"] == k, "correct_7b"].mean()), 2),
                         "3B_4bit_%": round(100 * float(rp.loc[rp["task"] == k, "correct_3b"].mean()), 2),
                         "hand_off_%": round(100 * float(ok_rp[(rp["task"] == k).to_numpy()].mean()), 2)} for k in TASK_ORDER])
se = round(standard_error(rp, "correct_7b"), 2)
print(f"Threshold {tau:.4f} (target on the choosing half: {target:.2f}%); one standard error of the six-task mean on the "
      f"reporting half: {se} points\n")
print(handoff.to_string(index=False))
print()
print(by_task.to_string(index=False))

Threshold 0.0000 (target on the choosing half: 38.07%); one standard error of the six-task mean on the reporting half: 0.8 points

                  set                          method  share_by_3b_%  accuracy_%  s_per_question
       reporting half                        7B alone            0.0      39.313           4.825
       reporting half               3B alone (4 bits)          100.0      45.040           1.719
       reporting half              hand-off at 0.0000          100.0      45.040           1.719
       reporting half perfect hand-off (either right)            NaN      54.984             NaN
reporting half, clean                        7B alone            0.0      39.200           4.825
reporting half, clean               3B alone (4 bits)          100.0      44.908           1.719
reporting half, clean              hand-off at 0.0000          100.0      44.908           1.719
reporting half, clean perfect hand-off (either right)            NaN      54.917             

In [4]:
# ── Cell 4: Four bits against fp16 (L2 run 5, if saved), memory and speed, expectation against result, files ──
fp16_ready = ((FP16_DIR / "per_question.csv").is_file()
              and not json.loads((FP16_DIR / "config.json").read_text(encoding="utf-8"))["smoke_test"])
if fp16_ready:
    fp = pd.read_csv(FP16_DIR / "per_question.csv", usecols=["id", "answer", "correct"]).set_index("id")
    assert len(fp) == 10675, "L2 run 5's saved answers aren't the full benchmark"
    q = pq.set_index("id")
    quant = pd.DataFrame([{"task": k, "fp16_%": round(100 * float(fp.loc[q.index[q["task"] == k], "correct"].astype(bool).mean()), 2),
                           "4bit_%": round(100 * float(q.loc[q["task"] == k, "correct_3b"].mean()), 2),
                           "same_answer_%": round(100 * float((fp.loc[q.index[q["task"] == k], "answer"].astype(str)
                                                               == q.loc[q["task"] == k, "answer"].astype(str)).mean()), 1)}
                          for k in TASK_ORDER])
    quant["change_points"] = (quant["4bit_%"] - quant["fp16_%"]).round(2)
    print(quant.to_string(index=False))
else:
    quant = None
    print("L2 run 5's full run (the 3B in fp16 on the benchmark) isn't saved yet: four bits against fp16 waits for it")

sp = {k: {a: b for a, b in v.items() if a != "answers"} for k, v in speed.items()}
both_gb = round(sp["3b_nf4"]["weights_gpu_gb"] + sp["7b_nf4"]["peak_gpu_gb"], 2)
hand = handoff[(handoff["set"] == "reporting half") & handoff["method"].str.startswith("hand-off")].iloc[0]
seven = handoff[(handoff["set"] == "reporting half") & (handoff["method"] == "7B alone")].iloc[0]
checks = [
    ("4-bit 3B weights about 2-3 GB", f"{sp['3b_nf4']['weights_gpu_gb']} GB", 1.5 <= sp["3b_nf4"]["weights_gpu_gb"] <= 3.5),
    ("4-bit 3B peak under 4 GB, one question at a time", f"{sp['3b_nf4']['peak_gpu_gb']} GB", sp["3b_nf4"]["peak_gpu_gb"] < 4),
    ("4-bit 7B peak about 6-7 GB at full size", f"{sp['7b_nf4']['peak_gpu_gb']} GB", 5.5 <= sp["7b_nf4"]["peak_gpu_gb"] <= 7.5),
    ("4-bit 3B slower than fp16 3B", f"{sp['3b_nf4']['s_per_question_mean']} s against {sp['3b_fp16']['s_per_question_mean']} s",
     sp["3b_nf4"]["s_per_question_mean"] > sp["3b_fp16"]["s_per_question_mean"]),
    ("4-bit 3B faster than 4-bit 7B", f"{t3} s against {t7} s", t3 < t7),
    ("overconfident at temperature 1: ECE above 0.05", f"ECE {calib['ece_T1']}", calib["ece_T1"] > 0.05),
    ("a temperature above 1 chosen", f"{T}", T > 1),
]
if quant is not None:
    worst = quant["change_points"].min()
    checks.append(("4 bits cost at most about 2 points on most tasks", f"changes {quant['change_points'].tolist()}",
                   int((quant["change_points"] >= -2).sum()) >= 4))
expect = pd.DataFrame(checks, columns=["expectation (23 Nov)", "result", "met"])
print(expect.to_string(index=False))

def md(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:            # tabulate isn't in the cse499 environment
        return "```\n" + df.to_string(index=False) + "\n```"

cal.to_csv(OUT / "calibration.csv", index=False)
rel.to_csv(OUT / "reliability.csv", index=False)
rc.to_csv(OUT / "risk_coverage.csv", index=False)
handoff.to_csv(OUT / "handoff.csv", index=False)
by_task.to_csv(OUT / "handoff_by_task.csv", index=False)
summary = {"temperature": T, "threshold": tau, "target_choosing_%": target, "standard_error_points": se, "calibration": calib,
           "both_loaded_gb": both_gb, "speed": sp, "expectations": [dict(zip(expect.columns, r)) for r in checks]}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, default=str), encoding="utf-8")
(OUT / "README.md").write_text("\n".join([
    f"# L3: the 4-bit 3B ({config['winner']}) and its hand-off to the 7B", "",
    "Worked out on the PC by `l3-handoff-calibration.ipynb` from the Kaggle run `l3-4bit-3b-handoff` (`../README.md`) and "
    "the new 7B's saved answers (L0 step 2), by the rule fixed on 23 Nov (in both notebooks' first cells). Chosen on the "
    "choosing half (md5 of the question ID even), reported on the other half.", "",
    "## The hand-off", "",
    f"Threshold {tau:.4f} on the confidence at temperature {T}. On the reporting half, the 3B answers "
    f"{hand['share_by_3b_%']:.1f}% of the questions; the mean of the six tasks' accuracies is {hand['accuracy_%']:.2f}% against "
    f"{seven['accuracy_%']:.2f}% for the 7B alone (one standard error: {se} points), at an expected {hand['s_per_question']:.2f} s "
    f"per question against {t7:.2f} s. With both models loaded on one GPU: about {both_gb} GB.", "",
    md(handoff), "", "By task, reporting half:", "", md(by_task), "",
    "## The confidence", "",
    f"- Expected calibration error (15 bins, reporting half): {calib['ece_T1']} at temperature 1, {calib['ece_chosen_T']} at {T}"
    + (" (the end of the list: a better temperature may lie beyond it)." if calib["at_list_end"] else "."),
    f"- Area under the risk-coverage curve: {calib['rc_auc_chosen_T']} (random order {calib['rc_auc_random']}, the best "
    f"possible {calib['rc_auc_best']}). Lower is better.", "",
    md(cal), "", "Error rate among the most confident answers (reporting half, chosen temperature):", "", md(rc), "",
    "## Memory and speed (one T4, one question at a time)", "",
    md(pd.DataFrame([{"model": k, **v} for k, v in sp.items()])), "",
    *(["## Four bits against fp16 (L2 run 5)", "", md(quant), ""] if quant is not None else []),
    "## Expectation (fixed 23 Nov) against result", "", md(expect), "",
    "## Files", "", "- `calibration.csv`, `reliability.csv`, `risk_coverage.csv`, `handoff.csv`, `handoff_by_task.csv`, "
    "`summary.json`",
]) + "\n", encoding="utf-8")
print(f"✓ Wrote {OUT}")

task  fp16_%  4bit_%  same_answer_%  change_points
 DTR   62.04   63.17           90.7           1.13
 BBR   16.65   16.55           64.4          -0.10
 BDC   49.79   48.17           88.1          -1.62
 DRE   46.63   48.72           89.2           2.09
 DSR   35.38   34.63           56.5          -0.75
 ORR   51.98   55.71           84.0           3.73
                            expectation (23 Nov)                                         result   met
                   4-bit 3B weights about 2-3 GB                                        2.42 GB  True
4-bit 3B peak under 4 GB, one question at a time                                        3.03 GB  True
         4-bit 7B peak about 6-7 GB at full size                                        8.64 GB False
                    4-bit 3B slower than fp16 3B                         1.719 s against 1.62 s  True
                   4-bit 3B faster than 4-bit 7B                        1.719 s against 4.825 s  True
  overconfident at temperature 

✓ Wrote d:\CSE499AB_project\Essential_task_delegation\Champion_Task\L3-results\full\L3\handoff
